#### Clasificación presencia ausencia

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier, ExtraTreesClassifier
from sklearn.metrics import classification_report, average_precision_score, precision_recall_curve, confusion_matrix
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import shap


c:\Users\rubar\miniconda3\envs\TFMenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.375, -60.0, -44.875]
min_time, max_time = pd.to_datetime("2013-01-01"), pd.to_datetime("2023-12-31")

In [3]:
fishing_ds = xr.open_dataset("../data/processed/dynamic/presence_HKP.nc")
fishing = fishing_ds["presence"]
fishing = fishing.fillna(0)

temp_ds = xr.open_dataset("../data/processed/dynamic/to_surface.nc")
temp = temp_ds["to"]
temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_lag1 = temp.shift(time=1).fillna(0)

temp_bottom_ds = xr.open_dataset("../data/processed/dynamic/temp_bottom.nc")
temp_bottom = temp_bottom_ds["to"]
temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

temp_bottom_lag1 = temp_bottom.shift(time=1).fillna(0)

chl_ds = xr.open_dataset("../data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

chl_lag1 = chl.shift(time=1).fillna(0)

mixed_ds = xr.open_dataset("../data/processed/dynamic/mixed_layer.nc")
mixed = mixed_ds["mlotst"]
mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

mixed_lag1 = mixed.shift(time=1).fillna(0)

depth_ds = xr.open_dataset("../data/processed/static/depth.nc")
depth = depth_ds["depth"] 
depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)
depth = depth.broadcast_like(temp)

zo_ds = xr.open_dataset("../data/processed/dynamic/zo_surface.nc")
zo = zo_ds["zo"]
zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)

so_ds = xr.open_dataset("../data/processed/dynamic/so_surface.nc")
so = so_ds["so"]
so = (so - so.mean()) / so.std()
so = so.fillna(0)

mask_ds = xr.open_dataset("../data/processed/static/fishing_area_mask.nc")
mask = mask_ds["mask"]
mask = mask.broadcast_like(temp)

month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)
month = month.broadcast_like(temp)

year = temp["time"].dt.year
year = (year - year.mean()) / year.std()
year = year.broadcast_like(temp)

lat = (temp["lat"] - temp["lat"].mean()) / temp["lat"].std()
lon = (temp["lon"] - temp["lon"].mean()) / temp["lon"].std()
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp) #se añade como dinámica porque ya se ha corregido la forma

temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, month, year, temp_lag1, mixed_lag1, temp_bottom_lag1, chl_lag1 = xr.align(temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, month, year, temp_lag1, mixed_lag1, temp_bottom_lag1, chl_lag1, join="inner")

cropped = lambda da: da.sel(
    lon=slice(min_lon, max_lon),
    lat=slice(min_lat, max_lat),
    time=slice(min_time, max_time)
)

temp = cropped(temp)
temp_bottom = cropped(temp_bottom)
chl = cropped(chl)
mixed = cropped(mixed)
mixed_lag1 = cropped(mixed_lag1)
fishing = cropped(fishing)
mask = cropped(mask)
depth = cropped(depth)
zo = cropped(zo)
so = cropped(so)
month = cropped(month)
month_sin = cropped(month_sin)
month_cos = cropped(month_cos)
lat = cropped(lat)
lon = cropped(lon)
year = cropped(year)
temp_lag1 = cropped(temp_lag1)
temp_bottom_lag1 = cropped(temp_bottom_lag1)
chl_lag1 = cropped(chl_lag1)


In [4]:
# Target
y = fishing

X = xr.Dataset({
    "temp": temp,
    "temp_bottom": temp_bottom,
    "chl": chl,
    "mixed": mixed,
    "depth": depth,
    "zo": zo,
    "so": so,
    "mask": mask,
    "month": month,
    "lat": lat,
    "lon": lon,
    "year": year,
    "temp_lag1": temp_lag1,
    "temp_bottom_lag1": temp_bottom_lag1,


})


time = X.time

train_time = time < np.datetime64("2020-01-01")  
test_time  = ~train_time

X_train_3d = X.sel(time=train_time)
X_test_3d  = X.sel(time=~train_time)

y_train_3d = y.sel(time=train_time)
y_test_3d  = y.sel(time=~train_time)


X_train_df = X_train_3d.to_dataframe().reset_index()
X_test_df  = X_test_3d.to_dataframe().reset_index()

# Convert target
y_train_df = y_train_3d.to_dataframe(name="target").reset_index()
y_test_df  = y_test_3d.to_dataframe(name="target").reset_index()

# Merge target with predictors
train_df = X_train_df.merge(
    y_train_df,
    on=["time", "lat", "lon"]
)

test_df = X_test_df.merge(
    y_test_df,
    on=["time", "lat", "lon"]
)

y_train = train_df["target"].values
y_test = test_df["target"].values

X_train_df = train_df.drop(columns=["target"])
X_test_df = test_df.drop(columns=["target"])


def add_spatial_features(df):
    df = df.copy()

    # cyclical month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # spatial interactions
    df["lat_lon"] = df["lat"] * df["lon"]
    # df["lat2"] = df["lat"] ** 2
    # df["lon2"] = df["lon"] ** 2

    # environmental interactions
    df["temp_depth"] = df["temp"] * df["depth"]
    df["chl_temp"] = df["chl"] * df["temp"]

    return df

X_train_df = add_spatial_features(X_train_df)
X_test_df = add_spatial_features(X_test_df)

X_train_df = X_train_df.drop(columns=["time"])
X_test_df = X_test_df.drop(columns=["time"])


In [5]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    class_weight={0:1, 1:2},
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train_df, y_train)

y_pred = rf.predict(X_test_df)
y_prob = rf.predict_proba(X_test_df)[:, 1]

prec, rec, thr = precision_recall_curve(y_test, y_prob)
f1 = 2 * (prec * rec) / (prec + rec)
best_idx = np.argmax(f1)
best_threshold = thr[best_idx]
y_pred_opt = (y_prob >= best_threshold).astype(int)
print("Best threshold:", best_threshold)
print("RandomForestClassifier")
print(classification_report(y_test, y_pred_opt))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

imp = pd.Series(rf.feature_importances_, index=X_train_df.columns)
imp.sort_values(ascending=False)

Best threshold: 0.386
RandomForestClassifier
              precision    recall  f1-score   support

           0       0.93      0.91      0.92      6113
           1       0.75      0.80      0.77      1951

    accuracy                           0.89      8064
   macro avg       0.84      0.86      0.85      8064
weighted avg       0.89      0.89      0.89      8064

PR-AUC: 0.8086731834995907
Confusion Matrix:
 [[5770  343]
 [ 637 1314]]


depth               0.127389
lon                 0.103972
lat_lon             0.081407
lat                 0.071330
temp_bottom_lag1    0.067812
zo                  0.062348
temp_bottom         0.062255
mixed               0.053996
mask                0.048071
temp_lag1           0.041882
temp_depth          0.041723
chl                 0.039329
temp                0.039328
so                  0.037189
month_cos           0.034348
chl_temp            0.033569
month               0.023167
year                0.021439
month_sin           0.009446
dtype: float64

In [6]:
hgb = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.05,
    max_iter=500,
    class_weight={0:1, 1:2},
    random_state=42
)

hgb.fit(X_train_df, y_train)

y_pred = hgb.predict(X_test_df)
y_prob = hgb.predict_proba(X_test_df)[:, 1]

prec, rec, thr = precision_recall_curve(y_test, y_prob)
f1 = 2 * (prec * rec) / (prec + rec)
best_idx = np.argmax(f1)
best_threshold = thr[best_idx]
y_pred_opt = (y_prob >= best_threshold).astype(int)
print("Best threshold:", best_threshold)
print("HistGradientBoostingClassifier")
print(classification_report(y_test, y_pred_opt))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# explainer = shap.Explainer(hgb.predict_proba, X_train)
# shap_values = explainer(X_test)
# shap.summary_plot(shap_values[:, :, 1], X_test)
# shap.plots.bar(shap_values[:, :, 1])
# shap.plots.scatter(shap_values[:, "temp_lag1", 1])




Best threshold: 0.4351757400333575
HistGradientBoostingClassifier
              precision    recall  f1-score   support

           0       0.95      0.88      0.91      6113
           1       0.69      0.86      0.77      1951

    accuracy                           0.87      8064
   macro avg       0.82      0.87      0.84      8064
weighted avg       0.89      0.87      0.88      8064

PR-AUC: 0.8296069156898852
Confusion Matrix:
 [[5437  676]
 [ 332 1619]]


In [7]:
lgbm = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0:1, 1:2},
    random_state=42
)

lgbm.fit(X_train_df, y_train)
y_pred = lgbm.predict(X_test_df)
y_prob = lgbm.predict_proba(X_test_df)[:, 1]

prec, rec, thr = precision_recall_curve(y_test, y_prob)
f1 = 2 * (prec * rec) / (prec + rec)
best_idx = np.argmax(f1)
best_threshold = thr[best_idx]
print("Best threshold:", best_threshold)
print("LGBMClassifier")
y_pred_opt = (y_prob >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_opt))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



[LightGBM] [Info] Number of positive: 2976, number of negative: 11136
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000769 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2963
[LightGBM] [Info] Number of data points in the train set: 14112, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.348315 -> initscore=-0.626456
[LightGBM] [Info] Start training from score -0.626456
Best threshold: 0.22379225640228506
LGBMClassifier
              precision    recall  f1-score   support

           0       0.94      0.90      0.92      6113
           1       0.72      0.81      0.76      1951

    accuracy                           0.88      8064
   macro avg       0.83      0.85      0.84      8064
weighted avg       0.88      0.88      0.88      8064

PR-AUC: 0.815829706497418
Confusion Matrix:
 [[5643  470]
 [ 487 1464]]


In [8]:
base_models = [
    ('hgb', HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.05,
        max_iter=500,
        class_weight='balanced',
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=400,
        class_weight={0:1, 1:2},
        random_state=42
    )),
    ('et', ExtraTreesClassifier(
        n_estimators=400,
        class_weight='balanced',
        random_state=42
    ))
]

stack = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)


stack.fit(X_train_df, y_train)
y_prob = stack.predict_proba(X_test_df)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, y_prob)

f1 = (2 * prec * rec) / (prec + rec + 1e-9)
best_idx = np.argmax(f1[:-1])
best_threshold = thr[best_idx]
y_pred = (y_prob >= best_threshold).astype(int)

print("Best threshold:", best_threshold)
print("Stacking Classifier: HistGradientBoostingClassifier + RandomForestClassifier + ExtraTreesClassifier")
print(classification_report(y_test, y_pred))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Best threshold: 0.22095850222501368
Stacking Classifier: HistGradientBoostingClassifier + RandomForestClassifier + ExtraTreesClassifier
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      6113
           1       0.67      0.73      0.70      1951

    accuracy                           0.85      8064
   macro avg       0.79      0.81      0.80      8064
weighted avg       0.85      0.85      0.85      8064

PR-AUC: 0.7201324012086823
Confusion Matrix:
 [[5426  687]
 [ 535 1416]]


In [9]:
cat = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=7,
    loss_function='Logloss',
    eval_metric='PRAUC',
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=200
)

cat.fit(X_train_df, y_train)

y_pred = cat.predict(X_test_df)
y_prob = cat.predict_proba(X_test_df)[:, 1]

prec, rec, thr = precision_recall_curve(y_test, y_prob)
f1 = 2 * (prec * rec) / (prec + rec)
best_idx = np.argmax(f1)
best_threshold = thr[best_idx]
y_pred_opt = (y_prob >= best_threshold).astype(int)
print("Best threshold:", best_threshold)
print("CatBoostClassifier")
print(classification_report(y_test, y_pred_opt))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

0:	learn: 0.8706463	total: 161ms	remaining: 2m 40s
200:	learn: 0.9559990	total: 2.02s	remaining: 8.03s
400:	learn: 0.9725897	total: 4.03s	remaining: 6.02s
600:	learn: 0.9820604	total: 6.09s	remaining: 4.04s
800:	learn: 0.9883270	total: 8.13s	remaining: 2.02s
999:	learn: 0.9922184	total: 10.1s	remaining: 0us
Best threshold: 0.5425161428970666
CatBoostClassifier
              precision    recall  f1-score   support

           0       0.95      0.89      0.92      6113
           1       0.71      0.85      0.77      1951

    accuracy                           0.88      8064
   macro avg       0.83      0.87      0.84      8064
weighted avg       0.89      0.88      0.88      8064

PR-AUC: 0.8271044400969498
Confusion Matrix:
 [[5374  739]
 [ 278 1673]]
